# Chapter 22 — Retries Are Side Effects Too

**Companion to *Applied AI*.**

An adapter appends one line to a file. The acknowledgement is lost. The
caller retries — and the line appears twice. This notebook first causes that
duplicate on purpose, then replays the preserved keyed run that fixes it, and
finally shows where the fix itself stops: six concurrent submissions, six
effects.

## Question

**When does trying again cause a second effect?**

## What this notebook does

It **demonstrates** the duplicate with a local file fixture (effect occurs,
acknowledgement lost, retry duplicates), then **reproduces** the pinned run
from `retry-rerun/2026-09-14-1b3c7a2/`: keyed replay, key collision refused,
authority re-checked on replay, cached failure, precondition drift, crash gap
— and the frozen concurrency diagnostic.

```text
retry  ≠  replay  ≠  duplicate
```

## Setup

Standard library only. No network, no API key, no `codeai` import.
Only bundle-relative paths are shown; override the evidence root with
`APPLIED_AI_EVIDENCE`. File effects below land in a temporary directory.

In [1]:
import json
import os
import tempfile
from pathlib import Path

def find_evidence_dir(marker="retry-rerun"):
    """Locate the preserved evidence. Override with APPLIED_AI_EVIDENCE."""
    env = os.environ.get("APPLIED_AI_EVIDENCE")
    if env and Path(env).expanduser().is_dir():
        return Path(env).expanduser()
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base / "evidence",
                     base / "experiments" / "applied-ai" / "evidence"):
            if (cand / marker).is_dir():
                return cand
    raise FileNotFoundError(
        "Preserved evidence not found. Set APPLIED_AI_EVIDENCE to the "
        "directory holding the Applied AI evidence bundles.")

EVIDENCE_DIR = find_evidence_dir()
BUNDLE = EVIDENCE_DIR / "retry-rerun" / "2026-09-14-1b3c7a2"
print("bundle: retry-rerun/2026-09-14-1b3c7a2")
results = json.loads((BUNDLE / "results.json").read_text(encoding="utf-8"))
print("cases:", sorted(results))

bundle: retry-rerun/2026-09-14-1b3c7a2
cases: ['authority-change', 'concurrency-probe', 'crash-after-effect', 'exact-duplicate', 'failed-replay', 'fingerprint-collision', 'precondition-drift', 'reopen-replay']


## 1. First, the failure: a retried computation vs a retried effect

Retrying a pure computation is safe — the worst case is wasted work.
Retrying a side effect re-causes it. A timeout says nothing about whether the
far side acted:

In [2]:
tmp = Path(tempfile.mkdtemp())
target = tmp / "line.txt"

class LostAck(Exception):
    """The effect happened; only the acknowledgement was lost."""

def append_line(path, text, lose_ack=False):
    with open(path, "a", encoding="utf-8") as f:
        f.write(text + "\n")
    if lose_ack:
        raise LostAck("timeout waiting for acknowledgement")
    return "ok"

# Effect occurs, acknowledgement is lost, caller observes uncertainty...
try:
    append_line(target, "notify reviewer", lose_ack=True)
except LostAck as e:
    print("caller sees:", e)

# ...caller retries, and the duplicate effect occurs:
append_line(target, "notify reviewer")
lines = target.read_text(encoding="utf-8").splitlines()
print("lines in file:", lines)
assert lines == ["notify reviewer", "notify reviewer"]
print()
print("Two identical lines from one intention. Retrying the *effect* is not")
print("retrying the *computation* — the file keeps both.")

caller sees: timeout waiting for acknowledgement
lines in file: ['notify reviewer', 'notify reviewer']

Two identical lines from one intention. Retrying the *effect* is not
retrying the *computation* — the file keeps both.


## 2. The fix: an operation identity the runtime can replay

The book's mechanism: the request carries an idempotency key, and the runtime
replays the recorded result for the *same* operation instead of acting again.
Same key, same request — one effect, two arrivals:

In [3]:
ex = results["exact-duplicate"]
print("statuses:", ex["statuses"])
print("reused  :", ex["reused"], "| effects:", ex["effects"], "| markers:", ex["markers"])
assert ex["statuses"] == ["succeeded", "succeeded"]
assert ex["effects"] == 1 and ex["markers"] == 1
print()
print("The second arrival returned the record. The adapter ran once.")
print("Reopen the ledger in a new process and the same key still replays")
print("(reopen-replay: %s, effects %d)." % (results["reopen-replay"]["reused"],
                                            results["reopen-replay"]["effects"]))

statuses: ['succeeded', 'succeeded']
reused  : a-ex1 | effects: 1 | markers: 1

The second arrival returned the record. The adapter ran once.
Reopen the ledger in a new process and the same key still replays
(reopen-replay: a-ro1, effects 1).


## 3. A key alone is not evidence

Three refusals that keep the key honest. A key matching a completed action
with a *different* request is a fingerprint collision, not a replay. A replay
under changed authority is denied, not served from the record. A precondition
that drifted since planning fails before any effect:

In [4]:
col = results["fingerprint-collision"]
print("collision:", col["conflict"])
print("error    :", col["error"])
print("effects  :", col["effects"])
assert col["conflict"] and col["effects"] == 1

auth = results["authority-change"]
print()
print("replay under revoked authority:", auth["status"], "| effects:", auth["effects"])
assert auth["status"] == "denied"

drift = results["precondition-drift"]
print("stale precondition            :", drift["status"], "| effects:", drift["effects"])
print("error                         :", drift["error"])
assert drift["status"] == "failed" and drift["effects"] == 0
print()
print("The key may replay only the same recorded operation — never bypass")
print("current authority, and never paper over a moved world.")

collision: True
error    : idempotency key 'k-co' matches completed action 'a-co1' but the request differs in: instruction
effects  : 1

replay under revoked authority: denied | effects: 1
stale precondition            : failed | effects: 0
error                         : precondition mismatch: expected state-A, observed state-B

The key may replay only the same recorded operation — never bypass
current authority, and never paper over a moved world.


## 4. A failure status never proves the effect did not happen

A cached failure replays FAILED with one effect, not zero: the write happened
and the connection reset after it. The crash gap is sharper — the effect
happened and the completion itself was recorded as failed. Both replay the
record instead of acting again:

In [5]:
fr = results["failed-replay"]
print("failed-replay   :", fr["statuses"], "| reused:", fr["reused"], "| effects:", fr["effects"])
assert fr["statuses"] == ["failed", "failed"] and fr["effects"] == 1

cr = results["crash-after-effect"]
print("crash-after-effect:", cr["statuses"], "| reused:", cr["reused"], "| effects:", cr["effects"])
print("  error:", cr["error"])
assert cr["effects"] == 1
print()
print("Chapter 19's rule from the other side: failure is not rollback, and a")
print("missing completion is a reason for reconciliation, not evidence that")
print("nothing happened.")

failed-replay   : ['failed', 'failed'] | reused: a-fa1 | effects: 1
crash-after-effect: ['failed', 'failed'] | reused: a-cr1 | effects: 1
  error: effect happened, completion recorded as failed

Chapter 19's rule from the other side: failure is not rollback, and a
missing completion is a reason for reconciliation, not evidence that
nothing happened.


## 5. Where the fix stops: six submissions, six effects

The frozen concurrency diagnostic fires two same-key submissions on each of
three keys at once. Sequential duplicates are safe (section 2). Concurrent
ones are not — both arrivals act before either record exists:

In [6]:
conc = results["concurrency-probe"]
total_effects = 0
for key in sorted(conc):
    outcomes = conc[key]["outcomes"]
    total_effects += conc[key]["effects"]
    print(f"{key}: {outcomes} -> effects={conc[key]['effects']}")
print("total: 6 submissions ->", total_effects, "effects")
assert total_effects == 6
assert all(conc[k]["effects"] == 2 for k in conc)
print()
print("PRESERVED NEGATIVE RESULT: there is no concurrency safety here. The")
print("key deduplicates sequential arrivals; it does not serialize concurrent")
print("ones. Six submissions caused six effects.")

k-conc-1: ['c1a: succeeded reused=None calls=1', 'c1b: succeeded reused=None calls=1'] -> effects=2
k-conc-2: ['c2a: succeeded reused=None calls=1', 'c2b: succeeded reused=None calls=1'] -> effects=2
k-conc-3: ['c3a: succeeded reused=None calls=1', 'c3b: succeeded reused=None calls=1'] -> effects=2
total: 6 submissions -> 6 effects

PRESERVED NEGATIVE RESULT: there is no concurrency safety here. The
key deduplicates sequential arrivals; it does not serialize concurrent
ones. Six submissions caused six effects.


## Interpretation

1. **Retrying a computation wastes work; retrying an effect re-causes it.**
   A timeout says nothing about whether the far side acted (section 1).
2. **Replay needs an operation identity plus a sameness check.** Same key and
   same request → return the record. Same key with a different request is a
   collision and is refused — the key alone is not evidence.
3. **Replay still passes today's gates.** Changed authority denies; drifted
   preconditions fail with zero effects.
4. **Failure status ≠ no effect.** Cached failures and the crash gap replay
   their records with the effect counted, not erased.
5. **Keys deduplicate; they do not serialize.** The concurrency probe's 6/6
   is the boundary of what this mechanism establishes.

## Try it yourself

1. In the section-1 fixture, add a keyed ledger (`dict[key, result]`) and
   re-run the lost-ack sequence. Which arrival appends, and which returns?
2. Change the retry in section 1 to a *different* line under the same key.
   Should the runtime replay or refuse? (Answer: refuse — section 3.)
3. Open the bundle's `fixture/conc-1.markers`: two markers, one key. What
   record would have had to exist — and when — for the second arrival to
   replay instead of act?

*Evidence: `experiments/applied-ai/evidence/retry-rerun/2026-09-14-1b3c7a2/`
(pinned run, `results.json` + ledgers + marker files). No network, no API
key, no `codeai` import.*